1. Check the lambda needed for the default code.
   JHUcase: 0.384
   JHUdeath: 0.149
   HHShospitalized: 0.247
2. Force adding option.

In [ ]:
from utils import *
warnings.filterwarnings("ignore")
lag_set = np.array([14, 7, 0])
report_freq_days = 1
lag_set = (lag_set / report_freq_days).astype(int)
state_list = state_abbr
group_size = len(lag_set)
train_period = 140
validation_period = 49
intercept_length = validation_period
test_period = 300
step_size = 30

dataset_list = ['JHUcase', 'JHUdeath', 'HHShospitalized']
for dataset in dataset_list:
    target_data = pd.read_csv('../data/' + dataset + '.txt', index_col = 0)
    target_data = target_data.loc[:, state_list].astype(float)
    if report_freq_days == 1:
        target_data = target_data.apply(moving_average_smoother).iloc[6:]
    target_dates = target_data.index
    state_data_dict = {}
    for state in state_list:
        if report_freq_days == 1:
            state_data = pd.read_csv(f'../data/{state}states.csv', index_col = 0, header = 0).astype(float).apply(moving_average_smoother).iloc[6:]
        elif report_freq_days == 7:
            state_data = pd.read_csv(f'../data/{state}states.csv', index_col = 0, header = 0).astype(float).rolling(window=7, min_periods=1).sum().iloc[6:]
        state_data_dict[state] = state_data[state_data.index.isin(target_dates)]
    all_codes = state_data_dict[state_list[0]].columns.to_numpy()
    U07_g_idx = np.where(all_codes == 'U07')[0][0]

    train_start_date_list = []
    validation_start_date_list = []
    test_start_date_list = []
    train_start_date = date_after(max(min(target_data.index), min(state_data_dict[state].index)), max(lag_set) * report_freq_days)
    # train_start_date = 20211101
    validation_start_date = date_after(train_start_date, train_period)
    test_start_date = date_after(validation_start_date, validation_period)
    while test_start_date < 20220901:
    # while test_start_date < 20230401:
        train_start_date_list.append(train_start_date)
        validation_start_date_list.append(validation_start_date)
        test_start_date_list.append(test_start_date)
        train_start_date = date_after(train_start_date, step_size)
        validation_start_date = date_after(train_start_date, train_period)
        test_start_date = date_after(validation_start_date, validation_period)

    data_handler = HealthDataHandler(state_list, lag_set, report_freq_days, target_data, state_data_dict)
    states_prev_W = None
    states_prev_g = None
    # selection = False
    for d_idx in range(len(train_start_date_list)):
        data_handler.get_X_y(start_date = train_start_date_list[d_idx], end_date = validation_start_date_list[d_idx], codes = all_codes)
        data_handler.get_test(start_test = validation_start_date_list[d_idx], codes = all_codes, period_length = validation_period)
        print(train_start_date_list[d_idx], '-', test_start_date_list[d_idx], end = ': ')
        check_stat_list = []
        for state in state_list:
            standardized_grouped_X_g = data_handler.standardized_grouped_X[state][:, U07_g_idx * group_size : U07_g_idx * group_size + group_size]
            check_stat = np.linalg.norm(standardized_grouped_X_g.T.dot(data_handler.standardized_y[state]), 2) / standardized_grouped_X_g.shape[0]
            check_stat_list.append(check_stat)
        print(np.min(check_stat))


In [ ]:
from utils_add import *
warnings.filterwarnings("ignore")
lag_set = np.array([14, 7, 0])
report_freq_days = 1
lag_set = (lag_set / report_freq_days).astype(int)
state_list = state_abbr

train_period = 140
validation_period = 49
intercept_length = validation_period
test_period = 300
# test_period = 180
step_size = 30

dataset_list = ['HHShospitalized']
# dataset_list = ['HHSflu']
method_list = ['grpLasso', 'forward', 'forward_backward']
strong_codes = ['U07']
for dataset in dataset_list:
    target_data = pd.read_csv('../data/' + dataset + '.txt', index_col = 0)
    target_data = target_data.loc[:, state_list].astype(float)
    if report_freq_days == 1:
        target_data = target_data.apply(moving_average_smoother).iloc[6:]
    target_dates = target_data.index
    state_data_dict = {}
    for state in state_list:
        if report_freq_days == 1:
            state_data = pd.read_csv(f'../data/{state}states.csv', index_col = 0, header = 0).astype(float).apply(moving_average_smoother).iloc[6:]
        elif report_freq_days == 7:
            state_data = pd.read_csv(f'../data/{state}states.csv', index_col = 0, header = 0).astype(float).rolling(window=7, min_periods=1).sum().iloc[6:]
        state_data_dict[state] = state_data[state_data.index.isin(target_dates)]
    all_codes = state_data_dict[state_list[0]].columns.to_numpy()
    strong_g_indices = np.where(np.isin(all_codes, strong_codes))[0]
    train_start_date_list = []
    validation_start_date_list = []
    test_start_date_list = []
    train_start_date = date_after(max(min(target_data.index), min(state_data_dict[state].index)), max(lag_set) * report_freq_days)
    # train_start_date = 20211101
    validation_start_date = date_after(train_start_date, train_period)
    test_start_date = date_after(validation_start_date, validation_period)
    while test_start_date < 20220901:
    # while test_start_date < 20230401:
        train_start_date_list.append(train_start_date)
        validation_start_date_list.append(validation_start_date)
        test_start_date_list.append(test_start_date)
        train_start_date = date_after(train_start_date, step_size)
        validation_start_date = date_after(train_start_date, train_period)
        test_start_date = date_after(validation_start_date, validation_period)

    data_handler = HealthDataHandler(state_list, lag_set, report_freq_days, target_data, state_data_dict)
    for method in method_list:
        phe_cnt_dict = {}
        # y_dict = {}
        # pred_dict = {}
        # pred_agg_dict = {}
        # date_plot_list = []
        # for state in state_list:
        #     y_dict[state] = []
        #     pred_dict[state] = []
        #     pred_agg_dict[state] = []
        # cnt_features = {}
        # dist_windows = {}
        # for state in state_list:
        #     cnt_features[state] = []
        #     dist_windows[state] = []
        
        states_prev_W = None
        states_prev_g = None
        # selection = False
        for d_idx in range(len(train_start_date_list)):
            data_handler.get_X_y(start_date = train_start_date_list[d_idx], end_date = validation_start_date_list[d_idx], codes = all_codes)
            data_handler.get_test(start_test = validation_start_date_list[d_idx], codes = all_codes, period_length = validation_period)
            print(train_start_date_list[d_idx], '-', test_start_date_list[d_idx])
            
            data_handler.run_all_single(intercept_length, method, alpha = 0.2, strong_g_indices = strong_g_indices, states_prev_g = states_prev_g, maxsteps = 4, states_prev_W = states_prev_W, tau = 0.5, M = 10, M_max = 30, fdr = 0.1)
            # for state in state_list:
            #     cnt_features[state].append(len(data_handler.active_g_indices[state]))
            #     if states_prev_g is not None:
            #         dist_windows[state].append(len(set(data_handler.active_g_indices[state]).symmetric_difference(set(states_prev_g[state]))))
        #     data_handler.run_all_agg(intercept_length, state_method = 'marginal_diff', auxiliary_state_list = None, nStates = 5, total_step = 10, selection = selection)
        
            for state in state_list:
                for g in data_handler.active_g_indices[state]:
                    if all_codes[g] in phe_cnt_dict:
                        phe_cnt_dict[all_codes[g]] += 1
                    else:
                        phe_cnt_dict[all_codes[g]] = 1
            
            top_5 = sorted(phe_cnt_dict, key=phe_cnt_dict.get, reverse=True)[:5]
            for key in top_5:
                print(f"{key}: {phe_cnt_dict[key]}")
            print()
            
            if method == 'derandomKnock':
                states_prev_W = data_handler.states_curr_W
            states_prev_g = data_handler.active_g_indices
        #     filename = f'test0215/{dataset}_{method}_{d_idx}.pkl'
        #     with open(filename, 'wb') as file:
        #         pickle.dump(data_handler, file)
            
        #     data_handler.get_test(start_test = test_start_date_list[d_idx], codes = all_codes, period_length = test_period)
        #     date_plot_list.append([datetime.strptime(str(d), '%Y%m%d') for d in data_handler.test_dates])
        #     for state in state_list:
        #         y_dict[state].append(data_handler.y_test[state])
        #         pred_dict[state].append(np.dot(data_handler.grouped_X_test[state], data_handler.recovered_coef[state]) + data_handler.recovered_intercept[state])
        #         pred_agg_dict[state].append(np.dot(data_handler.grouped_X_test[state], data_handler.recovered_coef_agg[state]) + data_handler.recovered_intercept_agg[state])

        #     dfout = {
        #         'date': date_plot_list[d_idx],
        #         **{f'y_{state}': y_dict[state][d_idx] for state in state_list},
        #         **{f'pred_{state}': pred_dict[state][d_idx] for state in state_list},
        #         **{f'pred_agg_{state}': pred_agg_dict[state][d_idx] for state in state_list}
        #     }
        #     dfout = pd.DataFrame(dfout)
        #     dfout.to_csv(f'test0215/{dataset}_{method}_{d_idx}.csv', index=False)
        # # for state in state_list:
        # #     plt.figure(figsize=(10, 6))
        # #     for d_idx in range(len(train_start_date_list)):
        # #         plt.plot(date_plot_list[d_idx], y_dict[state][d_idx], c = 'k', lw = 2)
        # #         plt.plot(date_plot_list[d_idx], pred_dict[state][d_idx], alpha = 0.8, lw = 2)
        # #     plt.title(state + ', single')
        # #     plt.xticks(rotation = 30)
        # #     plt.figure(figsize=(10, 6))
        # #     for d_idx in range(len(train_start_date_list)):
        # #         plt.plot(date_plot_list[d_idx], y_dict[state][d_idx], c = 'k', lw = 2)
        # #         plt.plot(date_plot_list[d_idx], pred_agg_dict[state][d_idx], alpha = 0.8, lw = 2)
        # #     plt.title(state + ', aggregated')
        # #     plt.xticks(rotation = 30)
        

In [ ]:
from utils_add import *
warnings.filterwarnings("ignore")
lag_set = np.array([14, 7, 0])
report_freq_days = 1
lag_set = (lag_set / report_freq_days).astype(int)
state_list = state_abbr

train_period = 140
validation_period = 49
intercept_length = validation_period
test_period = 300
# test_period = 180
step_size = 30

dataset_list = ['JHUcase', 'JHUdeath', 'HHShospitalized']
# dataset_list = ['HHSflu']
method_list = ['grpLasso', 'forward', 'forward_backward']
strong_codes = ['U07']
for dataset in dataset_list:
    target_data = pd.read_csv('../data/' + dataset + '.txt', index_col = 0)
    target_data = target_data.loc[:, state_list].astype(float)
    if report_freq_days == 1:
        target_data = target_data.apply(moving_average_smoother).iloc[6:]
    target_dates = target_data.index
    state_data_dict = {}
    for state in state_list:
        if report_freq_days == 1:
            state_data = pd.read_csv(f'../data/{state}states.csv', index_col = 0, header = 0).astype(float).apply(moving_average_smoother).iloc[6:]
        elif report_freq_days == 7:
            state_data = pd.read_csv(f'../data/{state}states.csv', index_col = 0, header = 0).astype(float).rolling(window=7, min_periods=1).sum().iloc[6:]
        state_data_dict[state] = state_data[state_data.index.isin(target_dates)]
    all_codes = state_data_dict[state_list[0]].columns.to_numpy()
    strong_g_indices = np.where(np.isin(all_codes, strong_codes))[0]
    train_start_date_list = []
    validation_start_date_list = []
    test_start_date_list = []
    train_start_date = date_after(max(min(target_data.index), min(state_data_dict[state].index)), max(lag_set) * report_freq_days)
    # train_start_date = 20211101
    validation_start_date = date_after(train_start_date, train_period)
    test_start_date = date_after(validation_start_date, validation_period)
    while test_start_date < 20220901:
    # while test_start_date < 20230401:
        train_start_date_list.append(train_start_date)
        validation_start_date_list.append(validation_start_date)
        test_start_date_list.append(test_start_date)
        train_start_date = date_after(train_start_date, step_size)
        validation_start_date = date_after(train_start_date, train_period)
        test_start_date = date_after(validation_start_date, validation_period)

    data_handler = HealthDataHandler(state_list, lag_set, report_freq_days, target_data, state_data_dict)
    for method in method_list:
        phe_cnt_dict = {}
        y_dict = {}
        pred_dict = {}
        pred_agg_dict = {}
        date_plot_list = []
        for state in state_list:
            y_dict[state] = []
            pred_dict[state] = []
            pred_agg_dict[state] = []
        # cnt_features = {}
        # dist_windows = {}
        # for state in state_list:
        #     cnt_features[state] = []
        #     dist_windows[state] = []
        
        states_prev_W = None
        states_prev_g = None
        selection = False
        for d_idx in range(len(train_start_date_list)):
            data_handler.get_X_y(start_date = train_start_date_list[d_idx], end_date = validation_start_date_list[d_idx], codes = all_codes)
            data_handler.get_test(start_test = validation_start_date_list[d_idx], codes = all_codes, period_length = validation_period)
            print(train_start_date_list[d_idx], '-', test_start_date_list[d_idx])
            
            data_handler.run_all_single(intercept_length, method, alpha = 0.2, strong_g_indices = strong_g_indices, states_prev_g = states_prev_g, maxsteps = 4, states_prev_W = states_prev_W, tau = 0.5, M = 10, M_max = 30, fdr = 0.1)
            # for state in state_list:
            #     cnt_features[state].append(len(data_handler.active_g_indices[state]))
            #     if states_prev_g is not None:
            #         dist_windows[state].append(len(set(data_handler.active_g_indices[state]).symmetric_difference(set(states_prev_g[state]))))
            data_handler.run_all_agg(intercept_length, state_method = 'marginal_diff', auxiliary_state_list = None, nStates = 5, total_step = 10, selection = selection)
        
            for state in state_list:
                for g in data_handler.active_g_indices[state]:
                    if all_codes[g] in phe_cnt_dict:
                        phe_cnt_dict[all_codes[g]] += 1
                    else:
                        phe_cnt_dict[all_codes[g]] = 1
            
            top_5 = sorted(phe_cnt_dict, key=phe_cnt_dict.get, reverse=True)[:5]
            for key in top_5:
                print(f"{key}: {phe_cnt_dict[key]}")
            print()
            
            if method == 'derandomKnock':
                states_prev_W = data_handler.states_curr_W
            states_prev_g = data_handler.active_g_indices
        #     filename = f'test0215/{dataset}_{method}_{d_idx}.pkl'
        #     with open(filename, 'wb') as file:
        #         pickle.dump(data_handler, file)
            
            data_handler.get_test(start_test = test_start_date_list[d_idx], codes = all_codes, period_length = test_period)
            date_plot_list.append([datetime.strptime(str(d), '%Y%m%d') for d in data_handler.test_dates])
            for state in state_list:
                y_dict[state].append(data_handler.y_test[state])
                pred_dict[state].append(np.dot(data_handler.grouped_X_test[state], data_handler.recovered_coef[state]) + data_handler.recovered_intercept[state])
                pred_agg_dict[state].append(np.dot(data_handler.grouped_X_test[state], data_handler.recovered_coef_agg[state]) + data_handler.recovered_intercept_agg[state])

            dfout = {
                'date': date_plot_list[d_idx],
                **{f'y_{state}': y_dict[state][d_idx] for state in state_list},
                **{f'pred_{state}': pred_dict[state][d_idx] for state in state_list},
                **{f'pred_agg_{state}': pred_agg_dict[state][d_idx] for state in state_list}
            }
            dfout = pd.DataFrame(dfout)
            dfout.to_csv(f'test0215/{dataset}_{method}_{d_idx}.csv', index=False)
        # for state in state_list:
        #     plt.figure(figsize=(10, 6))
        #     for d_idx in range(len(train_start_date_list)):
        #         plt.plot(date_plot_list[d_idx], y_dict[state][d_idx], c = 'k', lw = 2)
        #         plt.plot(date_plot_list[d_idx], pred_dict[state][d_idx], alpha = 0.8, lw = 2)
        #     plt.title(state + ', single')
        #     plt.xticks(rotation = 30)
        #     plt.figure(figsize=(10, 6))
        #     for d_idx in range(len(train_start_date_list)):
        #         plt.plot(date_plot_list[d_idx], y_dict[state][d_idx], c = 'k', lw = 2)
        #         plt.plot(date_plot_list[d_idx], pred_agg_dict[state][d_idx], alpha = 0.8, lw = 2)
        #     plt.title(state + ', aggregated')
        #     plt.xticks(rotation = 30)
        